In [2]:
import numpy as np
import pandas as pd
import glob
from numba import guvectorize
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import os
from datetime import datetime, timezone
import re
import json
import awkward as ak
import pygama 


from dbetto import Props
from legendmeta import LegendMetadata
from lgdo import lh5
import lgdo
from dspeed.vis.waveform_browser import WaveformBrowser
from pygama.pargen.dsp_optimize import run_one_dsp
import lgdo.types as lgdo_types
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
import matplotlib.patches as mpatches
from collections import defaultdict
from pathlib import Path

import itertools
from tqdm import tqdm
import time

In [3]:
data_path = "/global/cfs/cdirs/m2676/data/lngs/l200/public/prodenv/prod-blind/ref/latest"
config = Props.read_from(os.path.join(data_path, "config.json"), subst_pathvar=True)["setups"]["l200"]["paths"]
lmeta  = LegendMetadata(config["metadata"])
chmap = ak.Array(lmeta.channelmap(lmeta.dataprod.runinfo.p03.r000.phy.start_key).group("system").geds.values())

In [9]:
detector = chmap.name[0]
chn = chmap.daq.rawid[chmap.name == detector][0]

In [11]:
pet_files = sorted(glob.glob(f"{config['tier_pet']}/phy/l200-p0[346789]-*-phy-tier_pet.lh5"))


In [13]:
start = time.time()
data_init = lh5.read_as("/evt", pet_files, field_mask=["coincident", "geds", "trigger"], library="ak")
print("Took", time.time() - start, "s to read")

Took 89.8880660533905 s to read


In [32]:
tcm_files = sorted(glob.glob(f"{config["tier_tcm"]}/cal/p03/r000/*.lh5"))
raw_files = sorted(glob.glob(f"{config["tier_raw"]}/cal/p03/r000/*.lh5"))
